In [ ]:
# ============================================================
# RF Leakage-Risk Classification Workflow
# ============================================================
# This script trains a Random Forest classifier to distinguish
# non-secure leakage-risk regimes: low, high, and extreme leakage.
#
# Workflow:
# 1. Load processed CMG-GEM simulation dataset
# 2. Classify leakage fraction into risk categories
# 3. Exclude secure cases for leakage-relevant ML classification
# 4. Apply stratified train/test split
# 5. Apply manual oversampling only to the training set
# 6. Train Random Forest classifier
# 7. Evaluate performance on unchanged hold-out test set
# 8. Save metrics, confusion matrix, and figure outputs
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)


# ============================================================
# USER SETTINGS
# ============================================================

# Update this path according to your local or repository structure.
DATA_PATH = Path("data/example_dataset.csv")

OUTPUT_DIR = Path(".")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Leakage-fraction thresholds.
# Leakage fraction is expressed as a decimal fraction, not percent.
# secure: LF <= 0.01%
# low leakage: 0.01% < LF <= 1.805%
# high leakage: 1.805% < LF <= 3.6%
# extreme leakage: LF > 3.6%
SECURE_MAX = 0.0001      # 0.01%
MID_POINT = 0.01805      # 1.805%
EXTREME_MIN = 0.036      # 3.6%

# Input variables used for ML classification.
X_COLS = [
    "PERM_permeability",
    "PERM_perm_factor_of_failed_well",
    "PERM_dist_to_well_in_grid",
    "PERM_caprock_perm",
    "porosity",
    "aquifer_porosity",
    "aquifer_permeability",
]

# Pretty labels used only for tables and plots.
CLASS_LABEL_MAP = {
    "low_risk": "Low",
    "high_risk": "High",
    "extreme": "Extreme",
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def classify_leakage_fraction(frac: float) -> str:
    """
    Classify leakage fraction into four leakage-risk categories.

    Parameters
    ----------
    frac : float
        Leakage fraction expressed as a decimal fraction, not percent.

    Returns
    -------
    str
        One of: secure, low_risk, high_risk, extreme.
    """
    if pd.isna(frac):
        return np.nan

    if frac <= SECURE_MAX:
        return "secure"
    elif frac <= MID_POINT:
        return "low_risk"
    elif frac <= EXTREME_MIN:
        return "high_risk"
    else:
        return "extreme"


def manual_oversample_low_high(
    train_df: pd.DataFrame,
    label_col: str = "leak_category",
    random_state: int = 42
) -> pd.DataFrame:
    """
    Apply manual oversampling only to the training set.

    Low- and high-leakage cases are duplicated once, while
    extreme-leakage cases are kept unchanged. The resulting
    training set is shuffled using a fixed random seed.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataframe before oversampling.
    label_col : str
        Name of the leakage-category column.
    random_state : int
        Random seed for shuffling.

    Returns
    -------
    pd.DataFrame
        Oversampled and shuffled training dataframe.
    """
    extreme_df = train_df[train_df[label_col] == "extreme"]
    low_df = train_df[train_df[label_col] == "low_risk"]
    high_df = train_df[train_df[label_col] == "high_risk"]

    low_oversampled = pd.concat([low_df, low_df], ignore_index=True)
    high_oversampled = pd.concat([high_df, high_df], ignore_index=True)

    train_oversampled = pd.concat(
        [extreme_df, low_oversampled, high_oversampled],
        ignore_index=True
    )

    train_oversampled = train_oversampled.sample(
        frac=1,
        random_state=random_state
    ).reset_index(drop=True)

    return train_oversampled


def check_required_columns(df: pd.DataFrame, required_cols: list) -> None:
    """
    Check that all required columns exist in the dataframe.
    """
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise ValueError(
            "The following required columns are missing from the dataset: "
            f"{missing_cols}"
        )


def print_class_distribution(df: pd.DataFrame, label_col: str, title: str) -> None:
    """
    Print class counts and proportions.
    """
    print(f"\n=== {title} ===")
    print("\nCounts:")
    print(df[label_col].value_counts(dropna=False))
    print("\nProportions:")
    print(df[label_col].value_counts(normalize=True, dropna=False))


# ============================================================
# LOAD DATASET
# ============================================================

print("\nLoading dataset...")
df = pd.read_csv(DATA_PATH)

print("Dataset loaded:", df.shape)
print(df.head())

required_cols = X_COLS + ["leakage_fraction"]
check_required_columns(df, required_cols)


# ============================================================
# LEAKAGE-RISK CLASSIFICATION
# ============================================================

df["leak_category"] = df["leakage_fraction"].apply(classify_leakage_fraction)

print_class_distribution(
    df=df,
    label_col="leak_category",
    title="Full dataset leakage-category distribution"
)

# Save full class distribution.
full_distribution = pd.DataFrame({
    "count": df["leak_category"].value_counts(dropna=False),
    "proportion": df["leak_category"].value_counts(normalize=True, dropna=False)
})

full_distribution.to_csv(
    TABLES_DIR / "Full_Dataset_LeakageCategory_Distribution.csv",
    index=True
)


# ============================================================
# LEAKAGE-RELEVANT DATASET FOR MACHINE LEARNING
# ============================================================

df_leakage = df[
    df["leak_category"].isin(["low_risk", "high_risk", "extreme"])
].copy()

df_leakage.reset_index(drop=True, inplace=True)

print("\nLeakage-relevant dataset shape:", df_leakage.shape)

print_class_distribution(
    df=df_leakage,
    label_col="leak_category",
    title="Leakage-relevant dataset distribution"
)


# ============================================================
# STRATIFIED TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    df_leakage,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_leakage["leak_category"]
)

print_class_distribution(
    df=train_df,
    label_col="leak_category",
    title="Original training-set distribution"
)

print_class_distribution(
    df=test_df,
    label_col="leak_category",
    title="Unchanged hold-out test-set distribution"
)


# ============================================================
# MANUAL OVERSAMPLING OF TRAINING SET
# ============================================================

train_df_os = manual_oversample_low_high(
    train_df=train_df,
    label_col="leak_category",
    random_state=RANDOM_STATE
)

print_class_distribution(
    df=train_df_os,
    label_col="leak_category",
    title="Oversampled training-set distribution"
)

# Save training/test distributions.
train_distribution = pd.DataFrame({
    "original_train_count": train_df["leak_category"].value_counts(),
    "oversampled_train_count": train_df_os["leak_category"].value_counts(),
    "test_count": test_df["leak_category"].value_counts()
})

train_distribution.to_csv(
    TABLES_DIR / "RF_Train_Test_Class_Distribution.csv",
    index=True
)


# ============================================================
# PREPARE FEATURES AND LABELS
# ============================================================

label_encoder = LabelEncoder()
label_encoder.fit(df_leakage["leak_category"])

print("\nClass mapping:")
for class_name, encoded_value in zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
):
    print(f"{class_name} -> {encoded_value}")

labels_raw = label_encoder.classes_
labels_pretty = [CLASS_LABEL_MAP.get(label, label) for label in labels_raw]

print("\nClass order used in metrics and confusion matrix:")
for raw, pretty in zip(labels_raw, labels_pretty):
    print(f"{raw} -> {pretty}")

X_train = train_df_os[X_COLS].values
y_train = label_encoder.transform(train_df_os["leak_category"])

X_test = test_df[X_COLS].values
y_test = label_encoder.transform(test_df["leak_category"])


# ============================================================
# TRAIN RANDOM FOREST CLASSIFIER
# ============================================================

rf = RandomForestClassifier(
    n_estimators=350,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("\nTraining Random Forest classifier...")
rf.fit(X_train, y_train)

print("Training completed.")


# ============================================================
# EVALUATE ON UNCHANGED HOLD-OUT TEST SET
# ============================================================

y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

report_dict = classification_report(
    y_test,
    y_pred,
    target_names=labels_pretty,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

metrics_table = report_df.loc[
    labels_pretty + ["macro avg", "weighted avg"],
    ["precision", "recall", "f1-score", "support"]
].copy()

accuracy_row = pd.DataFrame(
    {
        "precision": [np.nan],
        "recall": [np.nan],
        "f1-score": [accuracy],
        "support": [len(y_test)]
    },
    index=["accuracy"]
)

metrics_table_exact = pd.concat([metrics_table, accuracy_row])

metrics_table_rounded = metrics_table_exact.copy()
metrics_table_rounded[["precision", "recall", "f1-score"]] = (
    metrics_table_rounded[["precision", "recall", "f1-score"]].round(3)
)

print("\n=== RF CLASSIFICATION METRICS ===")
print(f"Overall accuracy: {accuracy:.3f}")
print(f"Macro-average F1-score: {report_dict['macro avg']['f1-score']:.3f}")
print(f"Weighted-average F1-score: {report_dict['weighted avg']['f1-score']:.3f}")

print("\nClassification metrics:")
print(metrics_table_rounded.to_string())

for label in labels_pretty:
    print(
        f"{label} | "
        f"Precision: {report_dict[label]['precision']:.3f}, "
        f"Recall: {report_dict[label]['recall']:.3f}, "
        f"F1-score: {report_dict[label]['f1-score']:.3f}, "
        f"Support: {int(report_dict[label]['support'])}"
    )

metrics_table_exact.to_csv(
    TABLES_DIR / "Table_5_RF_ClassificationMetrics_exact.csv",
    index=True
)

metrics_table_rounded.to_csv(
    TABLES_DIR / "Table_5_RF_ClassificationMetrics_rounded.csv",
    index=True
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_pred)

print("\n=== CONFUSION MATRIX ===")
print(cm)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {label}" for label in labels_pretty],
    columns=[f"Predicted {label}" for label in labels_pretty]
)

cm_df.to_csv(
    TABLES_DIR / "Figure_5_RF_ConfusionMatrix_values.csv",
    index=True
)

fig, ax = plt.subplots(figsize=(4.2, 3.8), dpi=600)

hm = sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels_pretty,
    yticklabels=labels_pretty,
    annot_kws={"size": 11},
    square=True,
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Number of test cases"},
    ax=ax
)

ax.set_xlabel("Predicted class", fontsize=11)
ax.set_ylabel("True class", fontsize=11)
ax.set_title("Confusion Matrix", fontsize=12)

ax.tick_params(axis="x", labelsize=10, rotation=0)
ax.tick_params(axis="y", labelsize=10, rotation=0)

cbar = hm.collections[0].colorbar
cbar.ax.tick_params(labelsize=9)
cbar.set_label("Number of test cases", fontsize=10)

plt.tight_layout()

fig_path = FIGURES_DIR / "Figure_5_RF_ConfusionMatrix.png"
fig.savefig(fig_path, dpi=600, bbox_inches="tight")

plt.show()
plt.close(fig)

print(f"\nSaved confusion matrix figure: {fig_path}")


# ============================================================
# SAVE MODEL INPUT INFORMATION
# ============================================================

input_info = pd.DataFrame({
    "input_variable": X_COLS
})

input_info.to_csv(
    TABLES_DIR / "RF_InputVariables.csv",
    index=False
)

print("\nSaved RF workflow outputs in:")
print(f"- Figures: {FIGURES_DIR.resolve()}")
print(f"- Tables:  {TABLES_DIR.resolve()}")
print("\nRF classification workflow completed successfully.")